In [ ]:
import sys
import os
from pathlib import Path

# Setup paths to find the src module
notebook_dir = Path(os.getcwd())
project_root = notebook_dir.parent if notebook_dir.name == 'src' else notebook_dir
sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
print(f"Python Path: {sys.path[:3]}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import json
from datetime import datetime
import matplotlib.pyplot as plt

# Check PyTorch and CUDA
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

In [ ]:
import sys
PROJECT_ROOT = Path.cwd().parents[1]
print(PROJECT_ROOT)

sys.path.insert(0, str(PROJECT_ROOT))

from src.model.detector import YOLOv1
from src.preprocessing.loaders import load_imagenet_iterable

print("✓ Successfully imported project modules")

In [ ]:
class Config:
    """Training configuration for Habrok"""
    
    # Model parameters
    NUM_CLASSES = 20  # Pascal VOC has 20 classes
    
    # Training parameters
    BATCH_SIZE = 64  # Adjust based on GPU memory (V100 can handle this)
    LEARNING_RATE = 1e-3
    EPOCHS = 90  # Standard ImageNet training (reduce for testing)
    OPTIMIZER = "Adam"  # or "SGD"
    
    # Data parameters
    TRAIN_N = None  # None = full dataset, or set a number for testing (e.g., 50000)
    VAL_N = None    # None = full dataset, or set a number for testing (e.g., 5000)
    NUM_WORKERS = 4  # For DataLoader (Habrok has good CPU cores)
    
    # Learning rate schedule
    USE_LR_SCHEDULER = True
    LR_STEP_SIZE = 30
    LR_GAMMA = 0.1
    
    # Checkpointing
    CHECKPOINT_DIR = project_root / "checkpoints_detector"
    SAVE_FREQUENCY = 5  # Save every N epochs
    
    # Device
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Mixed precision training (faster on newer GPUs)
    USE_AMP = True  # Automatic Mixed Precision
    
    # Logging
    LOG_DIR = project_root / "logs"
    
    def __init__(self):
        self.CHECKPOINT_DIR.mkdir(exist_ok=True)
        self.LOG_DIR.mkdir(exist_ok=True)
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        
    def save_config(self):
        """Save configuration to JSON"""
        config_dict = {k: str(v) if isinstance(v, Path) else v 
                      for k, v in self.__dict__.items() 
                      if not k.startswith('_')}
        config_file = self.LOG_DIR / f"config_{self.timestamp}.json"
        with open(config_file, 'w') as f:
            json.dump(config_dict, f, indent=2)
        print(f"✓ Config saved to {config_file}")

config = Config()
config.save_config()

print(f"\n{'='*60}")
print(f"TRAINING CONFIGURATION")
print(f"{'='*60}")
print(f"Device: {config.DEVICE}")
print(f"Batch Size: {config.BATCH_SIZE}")
print(f"Learning Rate: {config.LEARNING_RATE}")
print(f"Epochs: {config.EPOCHS}")
print(f"Optimizer: {config.OPTIMIZER}")
print(f"Mixed Precision: {config.USE_AMP}")
print(f"Checkpoint Dir: {config.CHECKPOINT_DIR}")
print(f"{'='*60}\n")

In [ ]:
def get_data_loaders(config):
    """
    Load ImageNet data using the project's loader
    """
    print("Loading ImageNet dataset...")
    
    # Load the iterable datasets
    ds_dict = load_imagenet_iterable()
    
    # Create DataLoaders
    train_loader = DataLoader(
        ds_dict["train"],
        batch_size=config.BATCH_SIZE,
        num_workers=config.NUM_WORKERS,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    val_loader = DataLoader(
        ds_dict["validation"],
        batch_size=config.BATCH_SIZE,
        num_workers=config.NUM_WORKERS,
        pin_memory=True if torch.cuda.is_available() else False
    )
    
    print(f"✓ DataLoaders created")
    print(f"  - Training batches per epoch: ~{1281167 // config.BATCH_SIZE}")
    print(f"  - Validation batches: ~{50000 // config.BATCH_SIZE}")
    
    return train_loader, val_loader

# Load the data
train_loader, val_loader = get_data_loaders(config)

In [ ]:
model = YOLOv1(num_classes=config.NUM_CLASSES).to(config.DEVICE)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: YOLOPretrain")
print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
if config.OPTIMIZER == "Adam":
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE)
elif config.OPTIMIZER == "SGD":
    optimizer = optim.SGD(
        model.parameters(), 
        lr=config.LEARNING_RATE, 
        momentum=0.9,
        weight_decay=5e-4  # L2 regularization
    )
else:
    raise ValueError(f"Unknown optimizer: {config.OPTIMIZER}")

# Learning rate scheduler
scheduler = None
if config.USE_LR_SCHEDULER:
    scheduler = optim.lr_scheduler.StepLR(
        optimizer, 
        step_size=config.LR_STEP_SIZE, 
        gamma=config.LR_GAMMA
    )
    print(f"✓ LR Scheduler: StepLR (step={config.LR_STEP_SIZE}, gamma={config.LR_GAMMA})")

# Gradient scaler for mixed precision
scaler = torch.cuda.amp.GradScaler() if config.USE_AMP else None

print(f"✓ Optimizer: {config.OPTIMIZER}")
print(f"✓ Mixed Precision: {config.USE_AMP}")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, config, epoch):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{config.EPOCHS} [Train]")
    
    for batch_idx, batch in enumerate(pbar):
        # Extract data
        inputs = batch['image'].to(config.DEVICE)
        targets = batch['label'].to(config.DEVICE)
        
        optimizer.zero_grad()
        
        # Mixed precision training
        if config.USE_AMP:
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
        
        # Statistics
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        # Update progress bar
        if batch_idx % 10 == 0:
            pbar.set_postfix({
                'loss': f'{running_loss/total:.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc


def validate(model, loader, criterion, config, epoch):
    """Validate the model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}/{config.EPOCHS} [Val]")
    
    with torch.no_grad():
        for batch in pbar:
            inputs = batch['image'].to(config.DEVICE)
            targets = batch['label'].to(config.DEVICE)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            pbar.set_postfix({
                'loss': f'{running_loss/total:.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / total
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc


def save_checkpoint(model, optimizer, epoch, train_loss, val_loss, val_acc, config):
    """Save model checkpoint"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_acc': val_acc,
    }
    
    # Save latest
    latest_path = config.CHECKPOINT_DIR / "checkpoint_latest.pth"
    torch.save(checkpoint, latest_path)
    
    # Save periodic checkpoint
    if epoch % config.SAVE_FREQUENCY == 0:
        epoch_path = config.CHECKPOINT_DIR / f"checkpoint_epoch_{epoch:03d}.pth"
        torch.save(checkpoint, epoch_path)
        print(f"✓ Checkpoint saved: {epoch_path}")
    
    return latest_path


In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, scaler, config):
    """Main training loop"""
    
    # Training history
    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': [],
        'lr': []
    }
    
    best_val_acc = 0.0
    
    print(f"\n{'='*60}")
    print(f"Starting Training")
    print(f"{'='*60}\n")
    
    for epoch in range(1, config.EPOCHS + 1):
        # Training
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler, config, epoch
        )
        
        # Validation
        val_loss, val_acc = validate(
            model, val_loader, criterion, config, epoch
        )
        
        # Update learning rate
        current_lr = optimizer.param_groups[0]['lr']
        if scheduler:
            scheduler.step()
        
        # Record history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        
        # Print epoch summary
        print(f"\nEpoch {epoch}/{config.EPOCHS} Summary:")
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
        print(f"  Learning Rate: {current_lr:.6f}")
        
        # Save checkpoint
        save_checkpoint(model, optimizer, epoch, train_loss, val_loss, val_acc, config)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_path = config.CHECKPOINT_DIR / "checkpoint_best.pth"
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
            }, best_path)
            print(f"  ★ New best model! Val Acc: {val_acc:.2f}% (saved to {best_path})")
        
        print(f"{'='*60}\n")
    
    # Save training history
    history_path = config.LOG_DIR / f"training_history_{config.timestamp}.json"
    with open(history_path, 'w') as f:
        json.dump(history, f, indent=2)
    
    print(f"\n✓ Training completed!")
    print(f"✓ Best validation accuracy: {best_val_acc:.2f}%")
    print(f"✓ Training history saved to {history_path}")
    
    return history

# Run training
history = train_model(
    model, train_loader, val_loader, 
    criterion, optimizer, scheduler, scaler, 
    config
)

In [ ]:
def plot_training_history(history, config):
    """Plot training and validation metrics"""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss plot
    axes[0].plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], 'r-', label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=11)
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy plot
    axes[1].plot(epochs, history['train_acc'], 'b-', label='Train Acc', linewidth=2)
    axes[1].plot(epochs, history['val_acc'], 'r-', label='Val Acc', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=11)
    axes[1].grid(True, alpha=0.3)
    
    # Learning rate plot
    axes[2].plot(epochs, history['lr'], 'g-', linewidth=2)
    axes[2].set_xlabel('Epoch', fontsize=12)
    axes[2].set_ylabel('Learning Rate', fontsize=12)
    axes[2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[2].set_yscale('log')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save plot
    plot_path = config.LOG_DIR / f"training_plot_{config.timestamp}.png"
    plt.savefig(plot_path, dpi=150, bbox_inches='tight')
    print(f"✓ Training plot saved to {plot_path}")
    
    plt.show()

# Plot results
plot_training_history(history, config)

In [ ]:
#  Save final model weights only (smaller file)
final_model_path = project_root / "yolo_detector.pth"
torch.save(model.state_dict(), final_model_path)
print(f"\n✓ Final model saved to {final_model_path}")

# Print final statistics
print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"{'='*60}")
print(f"Best Validation Accuracy: {max(history['val_acc']):.2f}%")
print(f"Final Training Accuracy: {history['train_acc'][-1]:.2f}%")
print(f"Final Validation Accuracy: {history['val_acc'][-1]:.2f}%")
print(f"Model saved to: {final_model_path}")
print(f"{'='*60}")
